In [ ]:
!pip install torch opencv-python imageio mediapipe

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# import torchvision

import cv2
# import tensorflow as tf

import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio


In [ ]:
from processing.video_loader import VideoLoader
from processing.preprocessor import Preprocessor

# building data loading functions

download dataset from kaggle
https://www.kaggle.com/datasets/mohamedbentalb/lipreading-dataset and save them in data\lips-dataset

In [ ]:
test_path = r'.\data\lips-dataset\s1\bbaf2n.mpg'
video_loader = VideoLoader()
video_data = video_loader.load(test_path)
# video_data

In [ ]:
preprocessor = Preprocessor()
video_data = preprocessor.convert_to_greyscale(video_data)

if fail to crop lips, head over to google https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task



In [ ]:
# cv2.imshow("greyscale", video_data["frames"][0])
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [ ]:
char_to_num = {char: idx for idx, char in enumerate(vocab)}
num_to_char = {idx: char for char, idx in char_to_num.items()}

print(f"The vocabulary is: {vocab} (size = {len(vocab)})")

In [ ]:
def encode_text(text):
    return torch.tensor([char_to_num[c] for c in text], dtype=torch.long)

def decode_indices(indices):
    return "".join(num_to_char[int(i)] for i in indices)

x = encode_text("hello")
print(x)
print(decode_indices(x))

In [ ]:
def load_alignments(path:str) -> List[str]: 
    with open(path, 'r') as f: 
        lines = f.readlines() 
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': 
            tokens = [*tokens,' ',line[2]]
    return [encode_text(x) for x in tokens]

def load_data(path: str) -> dict: 
    # path = bytes.decode(path.numpy())
    #file_name = path.split('/')[-1].split('.')[0]
    # File name splitting for windows
    file_name = path.split('\\')[-1].split('.')[0]
    video_path = os.path.join('data','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data', 'lips-dataset', 'alignments','s1',f'{file_name}.align')
    video_data = video_loader.load(path) 
    alignments = load_alignments(alignment_path)
    video_data['alignments'] = alignments
    return video_data


video_data = load_data(test_path)
# print(video_data)
print(video_data['alignments'])
print([decode_indices(x) for x in video_data['alignments']])

# Creating data pipeline

In [ ]:
samples  = []
word_dict = {}
for i in os.listdir(r'.\data\lips-dataset\s1'):
    if i.endswith('.mpg'):
        print(i)
        video_data = load_data(os.path.join(r'.\data\lips-dataset\s1',i))
        samples.append((video_data['frames'], video_data['alignments']))
        
        print(video_data['alignments'])
        sentence = [decode_indices(x) for x in video_data['alignments']]
        print(sentence)
        for word in sentence: 
            if word not in word_dict: 
                word_dict[word] = 1
            else: 
                word_dict[word] += 1
    break

print(sorted(word_dict.items(), key=lambda x: x[1], reverse=True))

In [ ]:
cv2.imshow("test", video_data['frames'][40])
cv2.waitKey(0)
cv2.destroyAllWindows()

## Alignment Algorithm
### LBY: The cell below runs the align_videos.py as if it was in a terminal, input arguments include data directory and output directory
### It is important to note to save the kaggle dataset in its subfolder called data, of which should have the following subfolders:
### (a) alignments, (b) s1, (c) processedData [This is an addition]

In [7]:
!python align_videos.py --video-dir data/s1 --align-dir data/alignments/s1 --output-dir data/processedData --grayscale

[mpeg1video @ 0x1468390e0] ac-tex damaged at 22 17
[mpeg1video @ 0x1468390e0] Warning MVs not available

Done: 1000 videos aligned, 0 skipped. Output: data/processedData. Manifest: data/processedData/aligned_words.csv
